# Shared-core demo pipeline

**Purpose.** Run a **fixed batch** of three synthetic cases through `evaluate_batch` in **canonical_full_mode**, so gate outcomes, compensatory scores, and SCM abstention/fallback fields are all executed through the public API.

**What this shows.** The three cases are intentionally different: (1) all gates pass with typical feature values and adequate defaults for fallback; (2) a failed safety gate; (3) all gates pass but abstention triggers and fallback safety is below the canonical adequacy threshold, producing a rejection.

**Outputs (contract).** This notebook is solely responsible for:
- `outputs/figures/demo_pipeline_summary.txt`


In [1]:
from __future__ import annotations

import sys
from pathlib import Path


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in (start, *start.parents):
        if (p / "engine" / "corrected_public_engine_v1_1.py").is_file():
            return p
    raise RuntimeError(
        "Repository root not found (expected engine/corrected_public_engine_v1_1.py)."
    )


ROOT = repo_root()
sys.path.insert(0, str(ROOT / "engine"))
import corrected_public_engine_v1_1 as eng

CASES = [
    {
        "case_id": "demo_replay_pass",
        "features": {
            "intrinsic_safety": 0.62,
            "evidence_strength": 0.60,
            "bias_harm_index": 0.40,
            "uncertainty_calibration": 0.58,
            "traceability_integrity": 0.57,
        },
    },
    {
        "case_id": "demo_gate_fail",
        "features": {
            "intrinsic_safety": 0.35,
            "evidence_strength": 0.60,
            "bias_harm_index": 0.40,
            "uncertainty_calibration": 0.58,
            "traceability_integrity": 0.57,
        },
    },
    {
        "case_id": "demo_fullmode_abstention",
        "features": {
            "intrinsic_safety": 0.62,
            "evidence_strength": 0.60,
            "bias_harm_index": 0.40,
            "uncertainty_calibration": 0.42,
            "traceability_integrity": 0.50,
            "fallback_safety_delta": 0.15,
        },
    },
]

batch = eng.evaluate_batch(CASES, profile_names=["moderate"], mode=eng.MODE_CANONICAL_FULL)
approved = sum(
    1 for cid in batch if batch[cid]["profiles"]["moderate"]["approved"]
)
rejected = len(batch) - approved
digest = eng.hash_output(batch)

lines = [
    "Shared-core demo pipeline (03_demo_pipeline)",
    "============================================",
    "",
    "Purpose: This notebook runs a small, fixed batch of cases in canonical_full_mode,",
    "so both non-compensatory gates and SCM-derived abstention / fallback logic are",
    "exercised alongside the compensatory layer.",
    "",
    "How it works: Each case is a dict consumed by the engine's evaluate_batch helper.",
    "The moderate threshold profile from the engine's canonical table is used.",
    "",
    "Why this matters: The batch shows contrasting outcomes (approval with defaults,",
    "hard gate failure, and abstention with inadequate fallback) using the same public",
    "entry points the shared core documents.",
    "",
    "Output (this notebook only):",
    "- outputs/figures/demo_pipeline_summary.txt",
    "",
    f"Cases in batch:           {len(CASES)}",
    f"Approved (moderate):      {approved}",
    f"Rejected (moderate):      {rejected}",
    f"Batch digest (SHA-256):   {digest}",
    "",
    "Per-case governance outcomes (moderate profile):",
]
for c in CASES:
    r = batch[c["case_id"]]["profiles"]["moderate"]
    extra = ""
    if "abstention_rate" in r:
        extra = (
            f" abstention_rate={r['abstention_rate']} "
            f"abstention_triggered={r['abstention_triggered']} "
            f"fallback_adequate={r['fallback_adequate']}"
        )
    lines.append(
        f"- {c['case_id']}: {r['governance_outcome']} (approved={r['approved']}){extra}"
    )

out_path = ROOT / "outputs" / "figures" / "demo_pipeline_summary.txt"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Demo pipeline summary written.")


Demo pipeline summary written.
